# 3. Model configuration and training

Model construction is staged: select a model family and name, configure components or apply a preset, configure a dataset, compile, estimate resources, and train. Compilation creates the PyTorch module and dataset but marks the model untrained until `fit` succeeds.

In [ ]:
from pathlib import Path
import torch
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

device = "cuda" if torch.cuda.is_available() else "cpu"
wrapper = MSIAutoEncoderWrapper("tutorial_workspace", device=device)
image_path = Path("tutorial_workspace/imgs/example.imzML").resolve()
wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
wrapper.context_manager.set_binner("LinearBinning", str(image_path), bin_step=0.1)
wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner", str(image_path), max_bins=500, window_size=3
)
wrapper.workspace.set_active_image(str(image_path))

## Inspect the registries

A family determines the valid component categories, criteria, and runtime interface. Discovery is useful before writing a config and returns constructor metadata when `return_value=True`.

In [ ]:
wrapper.models_manager.get_available_model_types()
wrapper.models_manager.set_model_type("autoencoder", "tutorial-autoencoder")
wrapper.models_manager.get_available_component_categories()
wrapper.models_manager.get_available_model_presets()
wrapper.models_manager.get_available_criterions()
wrapper.models_manager.get_available_datasets()

## Use a preset, then override deliberately

`GradualReduction` derives input width from the active binner and estimates a convolution kernel from sampled peak envelopes. Parameters such as latent and projection dimensions remain explicit. The preset only fills the building buffer, so individual components can still be replaced before compilation. Registered names are preferred over classes or instances because names and parameters are portable JSON.

In [ ]:
wrapper.models_manager.set_model_preset(
    "GradualReduction",
    latent_dim=16,
    projection_dim=64,
)
wrapper.models_manager.set_dataset("PixelDataset", source="image")
model = wrapper.models_manager.compile_model(run_validation_pass=True)
print(model)

For complete manual construction, call `get_available_components(category)`, then `set_component(category, registered_name, **parameters)` for encoder, decoder, and optional projector/head. Custom components and presets are documented in [Custom models](../../../docs/CUSTOM_MODELS.md).

## Define training phases

Criteria are grouped by where they act. Reconstruction losses consume input and reconstruction; contrastive losses prepare augmented inputs and consume projection outputs; head losses are reserved for named head outputs. A phase may freeze direct child modules by name. Current lifecycle hooks are criterion hooks: phase-start precomputation and batch-start augmentation. No additional inter-layer training hook API is implied here.

In [ ]:
training_config = {
    "seed": 42,
    "phases": [
        {
            "phase_name": "joint_reconstruction_and_contrastive",
            "epochs": 20,
            "batch_size": 64,
            "freeze": [],
            "optimizer": {
                "type": "AdamW",
                "params": {"lr": 1e-3, "weight_decay": 1e-4},
            },
            "criterions": {
                "reconstruction": {
                    "mse": {
                        "target": "MSELoss",
                        "weight": 1.0,
                        "params": {},
                    },
                },
                "contrastive": {
                    "info_nce": {
                        "target": "InfoNCELoss",
                        "weight": 0.05,
                        "params": {
                            "temperature": 0.07,
                            "peak_sample_size": 512,
                            "peak_sample_seed": 42,
                        },
                    },
                },
            },
        }
    ],
}

`InfoNCELoss` samples spectra and peak envelopes once per phase, stores a bounded bank in the loaded model's transient training cache, injects sampled envelopes at batch start, and expands an `N` batch to `2N`. Clear the cache explicitly after changing data assumptions with `clear_training_cache()`. Detailed criterion contracts and the Masserstein loss are in [CRITERIONS.md](../../../docs/CRITERIONS.md).

## Estimate capacity before training

The estimator runs one evaluation forward probe and combines observed activation sizes with parameters, gradients, optimizer state, DataLoader buffering, known criterion workspaces, checkpoints, and history. Fractions in `(0, 1]` mean a fraction of currently available resources; larger values mean absolute bytes. It reports RAM, VRAM, and disk separately and can reduce batch size in a copied config. It is an estimate, not a guarantee: native reader caches, allocator fragmentation, OS activity, and future peaks are not fully measurable before training.

In [ ]:
report = wrapper.models_manager.estimate_training_resources(
    training_config,
    resource_limits={"ram": 0.75, "vram": 0.80, "disk": 0.90},
    auto_adjust_batch_size=True,
    safety_factor=1.25,
)
for phase in report["phases"]:
    print(
        phase["phase"],
        phase["recommended_batch_size"],
        phase["estimated_ram_bytes"],
        phase["estimated_vram_bytes"],
        phase["fits_limits"],
    )
print(report["estimated_disk_bytes"], report["disk_fits_limit"])
safe_training_config = report["recommended_training_config"]

Masserstein allocates matrices quadratic in the number of m/z bins; InfoNCE allocates similarity matrices quadratic in the expanded batch size. For either loss, keep the safety reserve and monitor the first epoch on the target machine.

In [ ]:
# Training may be expensive; run after reviewing the resource report.
# history = wrapper.models_manager.fit(safe_training_config)
# model_dir = wrapper.workspace.save_model(model_name="tutorial-autoencoder")